# Classification Model Evaluation — Decision Tree
**DevByte AI/ML Internship — BETA Project**

**Assigned:** 2026-06-30 &nbsp;|&nbsp; **Due:** 2026-07-28

### What this notebook does
Trains a Decision Tree classifier on a house property dataset, then evaluates it thoroughly using precision, recall, F1 score, and multiple confusion matrix layouts.

### Pipeline
| Step | Task |
|---|---|
| 1 | Setup & imports |
| 2 | Dataset generation |
| 3 | Exploratory Data Analysis |
| 4 | Preprocessing |
| 5 | Train default Decision Tree |
| 6 | Precision, Recall, F1 scores |
| 7 | Confusion matrix layouts (raw + normalized) |
| 8 | Decision tree visualization |
| 9 | Hyperparameter tuning (depth search) |
| 10 | Cross-validation |
| 11 | Feature importance |
| 12 | Summary |

---
## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})
SEED = 42

CLASS_NAMES  = ['Budget', 'Mid-Range', 'Premium']
CLASS_COLORS = ['#4C72B0', '#55A868', '#C44E52']

print('Libraries loaded successfully.')

---
## Step 2: Generate the Dataset

We reuse the same 9 house features from the regression task and convert the continuous price into three property classes:

| Class | Label | Price Range |
|---|---|---|
| 0 | Budget | Bottom third of prices |
| 1 | Mid-Range | Middle third |
| 2 | Premium | Top third |

In [ ]:
np.random.seed(SEED)
n = 1200

area_sqft          = np.random.randint(400, 6001, n)
num_bedrooms       = np.random.randint(1, 7, n)
num_bathrooms      = np.clip(num_bedrooms - np.random.randint(0, 2, n), 1, 5)
age_years          = np.random.randint(0, 41, n)
distance_km        = np.round(np.random.uniform(0.5, 25.0, n), 1)
has_garden         = np.random.choice([0, 1], n, p=[0.45, 0.55])
has_parking        = np.random.choice([0, 1], n, p=[0.30, 0.70])
neighborhood_score = np.round(np.random.uniform(1.0, 10.0, n), 1)
floor_number       = np.random.randint(0, 11, n)

noise = np.random.normal(0, 10, n)
price = (
      0.030  * area_sqft
    + 8.0    * num_bedrooms
    + 5.0    * num_bathrooms
    - 0.8    * age_years
    - 2.5    * distance_km
    + 10.0   * has_garden
    + 7.0    * has_parking
    + 6.0    * neighborhood_score
    - 0.5    * floor_number
    + noise
)
price = np.clip(price, 15, 600)

# Bin into 3 balanced classes using terciles
t1, t2 = np.percentile(price, [33.3, 66.6])
def label_class(p):
    if p < t1:  return 0   # Budget
    if p < t2:  return 1   # Mid-Range
    return 2                # Premium

target = np.array([label_class(p) for p in price])

df = pd.DataFrame({
    'area_sqft'          : area_sqft,
    'num_bedrooms'       : num_bedrooms,
    'num_bathrooms'      : num_bathrooms,
    'age_years'          : age_years,
    'distance_km'        : distance_km,
    'has_garden'         : has_garden,
    'has_parking'        : has_parking,
    'neighborhood_score' : neighborhood_score,
    'floor_number'       : floor_number,
    'property_class'     : target
})
df['property_label'] = df['property_class'].map({0:'Budget', 1:'Mid-Range', 2:'Premium'})

print('Dataset shape:', df.shape)
print('\nClass distribution:')
print(df['property_label'].value_counts())
df.head(8)

---
## Step 3: Exploratory Data Analysis

In [ ]:
# Class distribution bar + pie
class_counts = df['property_label'].value_counts().reindex(CLASS_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(CLASS_NAMES, class_counts.values, color=CLASS_COLORS, edgecolor='white', width=0.5)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Houses')

axes[1].pie(
    class_counts.values, labels=CLASS_NAMES,
    colors=CLASS_COLORS, autopct='%1.1f%%',
    startangle=120, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Class Distribution (Proportion)')

plt.suptitle('Property Class Balance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_class_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions by class
num_features = ['area_sqft', 'neighborhood_score', 'distance_km', 'age_years']
fig, axes    = plt.subplots(2, 2, figsize=(13, 8))

for ax, feat in zip(axes.flat, num_features):
    for label, col in zip(CLASS_NAMES, CLASS_COLORS):
        subset = df[df['property_label'] == label][feat]
        sns.kdeplot(subset, ax=ax, label=label, color=col, linewidth=2, fill=True, alpha=0.15)
    ax.set_title(f'{feat} by Class')
    ax.set_xlabel(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Property Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_by_class.png', bbox_inches='tight')
plt.show()

In [ ]:
# Box plots for area and neighborhood score by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

order = CLASS_NAMES
sns.boxplot(data=df, x='property_label', y='area_sqft',
            order=order, palette=CLASS_COLORS, linewidth=1.2, ax=axes[0])
axes[0].set_title('Area by Property Class')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Area (sqft)')

sns.boxplot(data=df, x='property_label', y='neighborhood_score',
            order=order, palette=CLASS_COLORS, linewidth=1.2, ax=axes[1])
axes[1].set_title('Neighborhood Score by Class')
axes[1].set_xlabel('Class'); axes[1].set_ylabel('Score')

plt.tight_layout()
plt.savefig('eda_boxplots.png', bbox_inches='tight')
plt.show()

---
## Step 4: Preprocessing

Split features from target, then split into 80% training and 20% testing using stratified sampling so each class is fairly represented in both sets.

In [ ]:
X = df.drop(columns=['property_class', 'property_label'])
y = df['property_class']
FEATURE_NAMES = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')
print(f'Features         : {len(FEATURE_NAMES)}')
print(f'\nTraining class counts:')
print(pd.Series(y_train).map({0:"Budget",1:"Mid-Range",2:"Premium"}).value_counts())

---
## Step 5: Train the Decision Tree Classifier

A Decision Tree learns a series of yes/no questions about the features (like "Is area > 2000 sqft?") and builds a tree of rules that leads to a predicted class.

We train two versions:
- **Default tree** (no depth limit, fully grown)
- **Pruned tree** (max depth found by tuning, avoids overfitting)

In [ ]:
# Default (unpruned) tree
dt_default = DecisionTreeClassifier(criterion='gini', random_state=SEED)
dt_default.fit(X_train, y_train)

y_pred_default = dt_default.predict(X_test)

print(f'Default Tree Depth  : {dt_default.get_depth()}')
print(f'Number of Leaves    : {dt_default.get_n_leaves()}')
print(f'Test Accuracy       : {accuracy_score(y_test, y_pred_default):.4f}')

---
## Step 6: Precision, Recall, and F1 Scores

**Precision** = Of all houses the model labeled as Premium, how many actually were Premium?

**Recall** = Of all houses that are actually Premium, how many did the model catch?

**F1 Score** = A single number that balances both Precision and Recall together.

All three are calculated for every class separately, and also as weighted averages.

In [ ]:
# Full classification report
report = classification_report(
    y_test, y_pred_default,
    target_names=CLASS_NAMES,
    digits=4
)
print('='*62)
print('         CLASSIFICATION REPORT — Default Decision Tree')
print('='*62)
print(report)

In [ ]:
# Per-class scores in a tidy DataFrame
precision_per_class = precision_score(y_test, y_pred_default, average=None)
recall_per_class    = recall_score(y_test, y_pred_default,    average=None)
f1_per_class        = f1_score(y_test, y_pred_default,        average=None)

metrics_df = pd.DataFrame({
    'Class'    : CLASS_NAMES,
    'Precision': np.round(precision_per_class, 4),
    'Recall'   : np.round(recall_per_class,    4),
    'F1 Score' : np.round(f1_per_class,        4),
})

# Add weighted averages row
weighted_row = pd.DataFrame([{
    'Class'    : 'Weighted Avg',
    'Precision': round(precision_score(y_test, y_pred_default, average='weighted'), 4),
    'Recall'   : round(recall_score(y_test,    y_pred_default, average='weighted'), 4),
    'F1 Score' : round(f1_score(y_test,        y_pred_default, average='weighted'), 4),
}])

metrics_df = pd.concat([metrics_df, weighted_row], ignore_index=True)
print(metrics_df.to_string(index=False))

In [ ]:
# Grouped bar chart of per-class precision, recall, F1
x      = np.arange(len(CLASS_NAMES))
width  = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
bars_p = ax.bar(x - width, precision_per_class, width, label='Precision', color='#4C72B0', edgecolor='white')
bars_r = ax.bar(x,         recall_per_class,    width, label='Recall',    color='#55A868', edgecolor='white')
bars_f = ax.bar(x + width, f1_per_class,        width, label='F1 Score',  color='#C44E52', edgecolor='white')

for bars in [bars_p, bars_r, bars_f]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.2f}',
            ha='center', va='bottom', fontsize=8
        )

ax.set_title('Precision, Recall, F1 Score per Class — Default Tree')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.12)
ax.legend()
plt.tight_layout()
plt.savefig('metrics_bar_default.png', bbox_inches='tight')
plt.show()

---
## Step 7: Confusion Matrix Layouts

A confusion matrix shows, for each true class, how many predictions landed in which predicted class. The diagonal shows correct predictions. Everything off the diagonal is a mistake.

In [ ]:
# Layout 1: Raw count confusion matrix (Seaborn heatmap style)
cm = confusion_matrix(y_test, y_pred_default)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 14, 'fontweight': 'bold'}, ax=ax
)
ax.set_title('Confusion Matrix — Raw Counts\n(Default Decision Tree)', pad=12)
ax.set_xlabel('Predicted Label', labelpad=10)
ax.set_ylabel('True Label', labelpad=10)
plt.tight_layout()
plt.savefig('cm_raw.png', bbox_inches='tight')
plt.show()

In [ ]:
# Layout 2: Normalized confusion matrix (row percentages — recall per class)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='YlOrRd',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    vmin=0, vmax=1,
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 12}, ax=ax
)
ax.set_title('Confusion Matrix — Row Normalized (% of True Class)\n(Default Decision Tree)', pad=12)
ax.set_xlabel('Predicted Label', labelpad=10)
ax.set_ylabel('True Label', labelpad=10)
plt.tight_layout()
plt.savefig('cm_normalized.png', bbox_inches='tight')
plt.show()

In [ ]:
# Layout 3: Side-by-side raw + normalized (publication style)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.8, linecolor='white', annot_kws={'size': 13}, ax=axes[0])
axes[0].set_title('Raw Counts')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='YlOrRd',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    vmin=0, vmax=1, linewidths=0.8, linecolor='white',
    annot_kws={'size': 12}, ax=axes[1])
axes[1].set_title('Row Normalized')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.suptitle('Confusion Matrix — Default Decision Tree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cm_side_by_side.png', bbox_inches='tight')
plt.show()

In [ ]:
# Layout 4: sklearn ConfusionMatrixDisplay (clean minimal style)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=axes[0], colorbar=True, cmap='Blues'
)
axes[0].set_title('sklearn Style — Raw Counts')

ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(
    ax=axes[1], colorbar=True, cmap='Greens',
    values_format='.2f'
)
axes[1].set_title('sklearn Style — Normalized')

plt.suptitle('Confusion Matrix — sklearn ConfusionMatrixDisplay', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cm_sklearn_style.png', bbox_inches='tight')
plt.show()

---
## Step 8: Decision Tree Visualization

One of the best things about Decision Trees is that you can actually see the rules it learned. Each box shows the question asked, how many samples reached that point, and what class is predicted there.

In [ ]:
# Shallow tree (depth 3) just for clean visualization
dt_viz = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=SEED)
dt_viz.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_viz,
    feature_names=FEATURE_NAMES,
    class_names=CLASS_NAMES,
    filled=True,
    rounded=True,
    impurity=True,
    proportion=False,
    fontsize=9,
    ax=ax
)
ax.set_title('Decision Tree Structure (max_depth=3 for clarity)', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('tree_visualization.png', bbox_inches='tight', dpi=150)
plt.show()
print('Blue nodes tend toward Budget, orange toward Mid-Range, green toward Premium.')

In [ ]:
# Text representation of the top 3 levels
print('Decision Tree Rules (top 3 levels):')
print('='*55)
print(export_text(dt_viz, feature_names=FEATURE_NAMES, max_depth=3))

---
## Step 9: Hyperparameter Tuning — Finding the Best Depth

A deeper tree memorizes the training data but fails on new data (overfitting). We search for the depth that gives the best balance between training and test performance.

In [ ]:
depths       = range(1, 21)
train_acc    = []
test_acc     = []
test_f1      = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, criterion='gini', random_state=SEED)
    dt.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, dt.predict(X_train)))
    preds = dt.predict(X_test)
    test_acc.append(accuracy_score(y_test, preds))
    test_f1.append(f1_score(y_test, preds, average='weighted'))

best_depth = list(depths)[np.argmax(test_f1)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curves
axes[0].plot(depths, train_acc, 'o-', color='royalblue', label='Train Accuracy', linewidth=2)
axes[0].plot(depths, test_acc,  's-', color='coral',     label='Test Accuracy',  linewidth=2)
axes[0].axvline(best_depth, color='green', linestyle='--', linewidth=1.4,
               label=f'Best depth = {best_depth}')
axes[0].set_title('Train vs Test Accuracy by Depth')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

# F1 curve
axes[1].plot(depths, test_f1, 'D-', color='mediumseagreen', linewidth=2)
axes[1].axvline(best_depth, color='red', linestyle='--', linewidth=1.4,
               label=f'Best depth = {best_depth}')
axes[1].set_title('Test Weighted F1 Score by Depth')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Weighted F1')
axes[1].legend()

plt.suptitle('Hyperparameter Search: max_depth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('depth_search.png', bbox_inches='tight')
plt.show()
print(f'Best max_depth: {best_depth}  |  Best Test F1: {max(test_f1):.4f}')

In [ ]:
# Train the tuned (pruned) tree
dt_tuned = DecisionTreeClassifier(max_depth=best_depth, criterion='gini', random_state=SEED)
dt_tuned.fit(X_train, y_train)
y_pred_tuned = dt_tuned.predict(X_test)

acc_tuned  = accuracy_score(y_test, y_pred_tuned)
f1_tuned   = f1_score(y_test, y_pred_tuned, average='weighted')
prec_tuned = precision_score(y_test, y_pred_tuned, average='weighted')
rec_tuned  = recall_score(y_test, y_pred_tuned, average='weighted')

print('='*50)
print(f'  Tuned Tree (max_depth={best_depth})')
print('='*50)
print(f'  Accuracy          : {acc_tuned:.4f}')
print(f'  Weighted Precision: {prec_tuned:.4f}')
print(f'  Weighted Recall   : {rec_tuned:.4f}')
print(f'  Weighted F1       : {f1_tuned:.4f}')
print('='*50)
print('\nFull Report:')
print(classification_report(y_test, y_pred_tuned, target_names=CLASS_NAMES, digits=4))

In [ ]:
# Confusion matrices: Default tree vs Tuned tree side by side
cm_tuned      = confusion_matrix(y_test, y_pred_tuned)
cm_tuned_norm = cm_tuned.astype('float') / cm_tuned.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.8, linecolor='white', annot_kws={'size': 13}, ax=axes[0])
axes[0].set_title(f'Default Tree (full depth)\nAccuracy = {accuracy_score(y_test,y_pred_default):.3f}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.8, linecolor='white', annot_kws={'size': 13}, ax=axes[1])
axes[1].set_title(f'Tuned Tree (max_depth={best_depth})\nAccuracy = {acc_tuned:.3f}')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.suptitle('Default vs Tuned Tree — Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cm_default_vs_tuned.png', bbox_inches='tight')
plt.show()

---
## Step 10: Cross-Validation

Instead of trusting a single 80/20 split, we split the data 5 different ways and train/test on each. This gives a more honest picture of how the model will perform on unseen data.

In [ ]:
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
models = {
    'Default Tree'          : DecisionTreeClassifier(criterion='gini', random_state=SEED),
    f'Tuned Tree (d={best_depth})': DecisionTreeClassifier(max_depth=best_depth, criterion='gini', random_state=SEED),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1_weighted')
    cv_results[name] = scores
    print(f'{name:<28} | Fold F1s: {np.round(scores, 3)} | Mean: {scores.mean():.4f} | Std: {scores.std():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
for i, (name, scores) in enumerate(cv_results.items()):
    color = CLASS_COLORS[i]
    ax.plot(range(1, 6), scores, 'o-', color=color, label=name, linewidth=2)
    ax.axhline(scores.mean(), color=color, linestyle='--', linewidth=1, alpha=0.6)

ax.set_title('5-Fold Stratified Cross-Validation — Weighted F1')
ax.set_xlabel('Fold'); ax.set_ylabel('Weighted F1 Score')
ax.set_xticks(range(1, 6))
ax.legend()
plt.tight_layout()
plt.savefig('cross_validation.png', bbox_inches='tight')
plt.show()

---
## Step 11: Feature Importance

The decision tree records how much each feature reduced the impurity (mixing of classes) across all its splits. Features used at the top of the tree or used many times get higher importance scores.

In [ ]:
importance_df = pd.DataFrame({
    'Feature'   : FEATURE_NAMES,
    'Importance': dt_tuned.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    importance_df['Feature'],
    importance_df['Importance'],
    color='#4C72B0', edgecolor='white'
)
for bar in bars:
    ax.text(
        bar.get_width() + 0.002,
        bar.get_y() + bar.get_height() / 2,
        f'{bar.get_width():.4f}',
        va='center', ha='left', fontsize=9
    )
ax.set_title(f'Feature Importance — Tuned Decision Tree (max_depth={best_depth})')
ax.set_xlabel('Gini Importance')
ax.set_xlim(0, importance_df['Importance'].max() * 1.2)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()
print(importance_df.sort_values('Importance', ascending=False).to_string(index=False))

---
## Step 12: Final Summary

In [ ]:
# Per-class metrics for both trees
prec_d = precision_score(y_test, y_pred_default, average=None)
rec_d  = recall_score(y_test,    y_pred_default, average=None)
f1_d   = f1_score(y_test,        y_pred_default, average=None)
prec_t = precision_score(y_test, y_pred_tuned,   average=None)
rec_t  = recall_score(y_test,    y_pred_tuned,   average=None)
f1_t   = f1_score(y_test,        y_pred_tuned,   average=None)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metric_pairs  = [(prec_d, prec_t, 'Precision'), (rec_d, rec_t, 'Recall'), (f1_d, f1_t, 'F1 Score')]
bar_colors_2  = ['#4C72B0', '#55A868', '#C44E52']

for ax, (default_vals, tuned_vals, metric_name) in zip(axes, metric_pairs):
    x     = np.arange(len(CLASS_NAMES))
    w     = 0.35
    ax.bar(x - w/2, default_vals, w, label='Default', color='#AECDE8', edgecolor='white')
    ax.bar(x + w/2, tuned_vals,   w, label='Tuned',   color='#2E75B6', edgecolor='white')
    ax.set_title(metric_name)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=10)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8)
    for i, (dv, tv) in enumerate(zip(default_vals, tuned_vals)):
        ax.text(i - w/2, dv + 0.02, f'{dv:.2f}', ha='center', fontsize=7.5)
        ax.text(i + w/2, tv + 0.02, f'{tv:.2f}', ha='center', fontsize=7.5)

plt.suptitle('Default vs Tuned Tree: Per-Class Metric Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('summary_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Final summary table
summary = pd.DataFrame([
    {
        'Model'          : 'Default Tree (full)',
        'Depth'          : dt_default.get_depth(),
        'Leaves'         : dt_default.get_n_leaves(),
        'Accuracy'       : round(accuracy_score(y_test, y_pred_default), 4),
        'W. Precision'   : round(precision_score(y_test, y_pred_default, average='weighted'), 4),
        'W. Recall'      : round(recall_score(y_test, y_pred_default, average='weighted'), 4),
        'W. F1'          : round(f1_score(y_test, y_pred_default, average='weighted'), 4),
    },
    {
        'Model'          : f'Tuned Tree (d={best_depth})',
        'Depth'          : dt_tuned.get_depth(),
        'Leaves'         : dt_tuned.get_n_leaves(),
        'Accuracy'       : round(acc_tuned, 4),
        'W. Precision'   : round(prec_tuned, 4),
        'W. Recall'      : round(rec_tuned, 4),
        'W. F1'          : round(f1_tuned, 4),
    },
])

print('='*80)
print('                    FINAL MODEL SUMMARY')
print('='*80)
print(summary.to_string(index=False))
print('='*80)
print(f'\nTop feature: {importance_df.sort_values("Importance",ascending=False).iloc[0]["Feature"]}')
print(f'Classes    : Budget (0), Mid-Range (1), Premium (2)')

---
## What Each Metric Means (Quick Reference)

| Metric | Formula | What it tells you |
|---|---|---|
| **Accuracy** | Correct / Total | Overall percentage of right answers |
| **Precision** | TP / (TP + FP) | Of all predicted as class X, how many actually were X |
| **Recall** | TP / (TP + FN) | Of all actual class X, how many did the model catch |
| **F1 Score** | 2 * P * R / (P + R) | Balanced score between Precision and Recall |
| **Weighted Avg** | Average weighted by class size | Overall score respecting class imbalance |

TP = True Positive, FP = False Positive, FN = False Negative